# Домашнее задание по теме «Метрические и линейные алгоритмы»



Для большинства алгоритмов классического машинного обучения давно существуют готовые реализации: мы уже использовали kNN из sklearn в семинарах. Это очень удобно для решения практических задач, потому что для получения результата достаточно нескольких строк кода. Но чтобы эффективно использовать такие модели, важно понимать их ограничения, сильные и слабые стороны — то, как они работают «под капотом».

В этом домашнем задании ты узнаешь детали реализации метрических алгоритмов:
- kNN
- Линейная регрессия

Как понять, что задание выполнено хорошо:

- все `# Напиши код здесь` заполнены, и код работает без ошибок;
- результаты твоих реализаций KNNClassifier и KNNRegressor точно совпадают с результатами из sklearn;
- даны чёткие, развёрнутые ответы на вопросы.

# Метрические модели

## Подготовка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_squared_error, r2_score
)

In [ ]:
def get_train_test(complex_function:callable, num_points=100, outliers=False, **kwargs):
    """ Генерация данных """
    np.random.seed(123)

    X = np.linspace(0, 10, num_points)
    X = X.reshape(-1, 1)

    X_train = X[::3] # Берём каждый третий элемент для обучения
    X_test = X[np.setdiff1d(np.arange(len(X)), np.arange(0, len(X), 3))] # Оставшиеся — для теста
    # Тут мы могли бы сгенерировать новый х: например, более плотно распределённые точки
    # Главное — что есть некоторые точки исходной функции, между которыми нужно заполнить пустое пространство

    # Значения функции с шумом
    y_train = complex_function(X_train).ravel() + np.random.normal(0, 0.2, len(X_train))
    y_test = complex_function(X_test).ravel() + np.random.normal(0, 0.2, len(X_test))

    if outliers:
        y_train = add_outliers(y_train, **kwargs)

    return X, X_train, X_test, y_train, y_test

def add_outliers(y,
                 num_outliers=None,
                 strength_std_devs=None,
                 random_seed=42):
    """ Добавление выбросов """

    if not num_outliers:
        num_outliers = int(len(y_train) * 0.1)

        if num_outliers == 0:
            num_outliers = 2

    if not strength_std_devs:
        strength_std_devs = 5

    y = np.array(y)
    y_with_outliers = y.copy()

    std_dev = np.std(y)

    if std_dev == 0:
        outlier_shift = strength_std_devs
    else:
        outlier_shift = strength_std_devs * std_dev

    rng = np.random.RandomState(seed=random_seed)

    # Случайно выбираем индексы выбросов
    outlier_indices = rng.choice(len(y), num_outliers, replace=False)

    for idx in outlier_indices:
        # Произвольно решаем, будет ли outlier больше или меньше текущего значения
        sign = rng.choice([-1, 1])
        y_with_outliers[idx] += sign * outlier_shift * (rng.rand() * 0.5 + 0.5)

    return y_with_outliers

def plot_data(X, X_train, X_test, y_train, y_test,
              complex_function:callable,
              pred=None,
              labels=[''],
              title='Синтетические данные для регрессии',
              subplot=None,
              highlight_train_range=None):
    """ Визуализация данных/результатов"""

    if not subplot:
        plt.figure(figsize=(12, 6))

    plt.scatter(X_train, y_train, c='black', alpha=0.5, label='Обучающая выборка')
    plt.scatter(X_test, y_test, c='gray', alpha=0.5, label='Тестовая выборка')
    plt.plot(X, complex_function(X), 'b-', label='Истинная функция')

    if pred is not None:
        for y_pred, label in zip(pred, labels):
            plt.plot(X_test, y_pred, alpha=1, label=label)

    if highlight_train_range:
        plt.axvspan(highlight_train_range[0], highlight_train_range[1],
                     color='lightgray', alpha=0.4, zorder=-1, label='Диапазон обучения')

    plt.grid(True)
    plt.legend()
    plt.title(title)

## Реализация собственного kNN

### Задача 1. Реализация BaseKNN [2 балла]

Класс ```BaseKNN``` — основа всех моделей. Его задачи:
* хранение обучающей выборки;
* реализация логики ключевого функционала kNN — поиска ближайших соседей (метод ```_find_k_nearest_neighbours```);
* вычисление весов на основе расстояний (метод ```_calculate_weights```).

Ниже представлен код с базовой структурой классов. Твоя задача — заполнить пропущенные части, руководствуясь комментариями и описаниями.

#### Задача 1.1. Функции ядра [0,5 балла]

Реализуй следующие функции ядра:

1. Гауссово — `_gaussian_kernel`:
$$\frac{1}{\sqrt{2\pi}}\exp(-\frac{1}{2}d^2).$$
2. Епанечникова — `_epanechnikov_kernel`:
$$\frac{3}{4}(1-d^2)\cdot I(|d|\le1).$$
3. Треугольное — `_triangular_kernel`:
$$(1-|d|)\cdot I(|d|\le1).$$
4. Прямоугольное — `_uniform_kernel`:
$$\frac{1}{2}I(|d|\le1).$$

#### Задача 1.2. Метод ```fit``` [0,5 балла]

1. Преобразуй $X$ и $Y$ в `np.array` с помощью функции `ensure_numpy`.
2. Убедись, что $X$ — двумерный массив (`n_points`, `n_features`).
3. Если $X$ одномерный (`n_points`), преобразуй его в (`n_samples`, 1): это необходимо для корректного вычисления расстояний.

#### Задача 1.3. Метод ```_find_k_nearest_neighbours``` [0,5 балла]

1. Вычисли расстояния от $x$ до всех точек в `self.X_train`, используя `np.linalg.norm (..., $axis=1$)`.
2. Найди индексы ближайших соседей, используя `np.argsort()`.
3. Найди расстояния до этих ближайших соседей.

#### Задача 1.4. Метод ```_calculate_weights``` [0,5 балла]

Примени взвешивание `inverse distance` для `weights == 'distance'`:
$$\frac{1}{\left|d\right|+\epsilon}.$$

В ином случае:
1. Масштабируй расстояния на `self.bandwidth` и `self.epsilon`.
2. Примени выбранную функцию ядра `self._weighting_function`.

> **Примечание.** Здесь $d$ — distances (`np.array`).

In [ ]:
from collections import defaultdict, Counter
import logging

def ensure_numpy(data):
    """ Функция для проверки типа входных данных """
    if isinstance(data, pd.DataFrame) or isinstance(data, pd.Series):
        return data.values
    return np.asarray(data)

class BaseKNN():
    def __init__(self, n_neighbours=5, weights='uniform', kernel='uniform', bandwidth=1.0, epsilon=1e-6):
        """ Базовый класс, который отвечает за запоминание выборки и поиск ближайших соседей """
        self.n_neighbours = n_neighbours
        self.weights = weights
        self.kernel = kernel
        self.bandwidth = bandwidth
        self.epsilon = epsilon
        self.X_train = None
        self.y_train = None

        self._weighting_function = self._get_kernel_function(self.kernel)

    def _gaussian_kernel(self, d):
        # Напиши код здесь
        pass

    def _epanechnikov_kernel(self, d):
        # Напиши код здесь
        pass

    def _triangular_kernel(self, d):
        # Напиши код здесь
        pass

    def _uniform_kernel(self, d):
        # Напиши код здесь
        pass

    def _get_kernel_function(self, kernel_name):
        """
        Возвращает функцию ядра в зависимости от выбранного типа
        """
        kernels = {
            'gaussian': self._gaussian_kernel,
            'epanechnikov': self._epanechnikov_kernel,
            'triangular': self._triangular_kernel,
            'uniform': self._uniform_kernel
        }

        return kernels.get(kernel_name)

    def fit(self, X, y):
        """
        Обучение kNN. Фактически - запоминание данных из обучающей выборки
        """

        # Напиши код здесь: запомни обучающие данные X и y в self.X_train и self.y_train
        return self

    def _find_k_nearest_neighbours(self, x):
        """
        Поиск k ближайших соседей для одной точки

        :param x: одна точка из тестовой выборки, shape (n_features)

        :return k_indices: индексы ближайших соседей из обучающей выборки
        :return k_distances: расстояния до каждого из k ближайших соседей
        """
        # Напиши код здесь: реализуй поиск k ближайших соседей
        distances = # Напиши код здесь
        k_indices = # Напиши код здесь
        k_distances = # Напиши код здесь
        return k_indices, k_distances


    def _calculate_weights(self, distances):
        """
        Подсчёт весов для одной точки. Масштабируем расстояния, используя
        параметр bandwidth, затем передаём их в функцию ядра.

        :param distances: массив расстояний от тестовой точки до других точек (n_features)

        :return: массив расстояний с применением функции ядра (n_features)
        """
        if self.weights == 'distance':
            weights = # Напиши код здесь
            return weights

        scaled_distances = # Напиши код здесь
        weighted_distances = # Напиши код здесь
        return weighted_distances


    def predict(self, X):
        """ Реализуем этот метод в дочерних классах """
        raise NotImplementedError("Не реализован в базовом kNN")


Основа готова. В ней реализован весь базовый функционал, необходимый kNN для решения любой задачи.

### Задача 2. KNNRegressor [3 балла]

Займёмся регрессией. Идея в следующем: предсказание новой точки — это среднее (или взвешенное среднее) значений её соседей.

Нужно:
 * реализовать обычный регрессор (```if self.weights == 'uniform':```),
 * протестировать его,
 * перейти к взвешенному.

> **Примечание.** Далее все указания будут приведены в комментариях.

#### Задача 2.1. Метод ```predict``` [1,5 балла]

Реализуй метод ```predict``` для регрессора.

#### Задача 2.2. Метод ```predict``` (взвешивание) [1,5 балла]

Реализуй взвешивание в методе  ```predict```: `if self.weights != 'uniform'`.

In [ ]:
class KNNRegressor(BaseKNN):
    def __init__(self, n_neighbours=5, weights='uniform', kernel='uniform', bandwidth=1.0, epsilon=1e-6):
        super().__init__(n_neighbours=n_neighbours,
                         weights=weights,
                         kernel=kernel,
                         bandwidth=bandwidth,
                         epsilon=epsilon)

    def predict(self, X):
        """
        Предсказывает значения для тестовой выборки X

        :param X: тестовые данные (n_test_points, n_features)

        :return y_pred: массив предсказанных для каждой точки значений (n_test_points)
        """
        pass
        # Преобразуй X в numpy.array по аналогии с методом .fit()
        # Убедись, что X — двумерный массив

        X = # Напиши код здесь

        # Инициализируй массив для предсказаний (n_test_points)
        # Значения в нём будут обновляться итеративно
        y_pred = # Напиши код здесь

        for i, x in enumerate(X):
            k_indices, k_distances = # Напиши код здесь: индексы и расстояния k ближайших соседей
            k_targets = # Напиши код здесь: значения целевой переменной для этих соседей

            if self.weights == 'uniform':
                # Это обычный kNN. Предсказание — среднее значение k_targets (или медиана)
                # Напиши код здесь
                pass
            else:
                # Взвешенный kNN. Предсказание — взвешенное среднее k_targets
                # Напиши код здесь
                pass

        return y_pred



Отлично! Теперь попробуй протестировать свою модель. Результаты должны сойтись с sklearn.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='target')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_reg = KNNRegressor(n_neighbours=5)
knn_reg_Sklearn = KNeighborsRegressor(n_neighbors=5)

knn_reg.fit(X_train_scaled, y_train)
knn_reg_Sklearn.fit(X_train_scaled, y_train)

y_pred_reg = knn_reg.predict(X_test_scaled)
y_pred_reg_Sklearn = knn_reg_Sklearn.predict(X_test_scaled)

print(f"Наша реализация R2 Score: {r2_score(y_test, y_pred_reg):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg):.3f}")
print(f"Sklearn R2 Score: {r2_score(y_test, y_pred_reg_Sklearn):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg_sklearn):.3f}")

In [ ]:
same = 0
different = 0
for i, j in zip(y_pred_reg, y_pred_reg_Sklearn):
    if i == j:
        same += 1
    else:
        different += 1
        print('different')

print(f'Совпадений: {same}')
print(f'Различий: {different}')

Теперь взвешивания.

In [ ]:
knn_reg = KNNRegressor(n_neighbours=5, weights='distance')
knn_reg_Sklearn = KNeighborsRegressor(n_neighbors=5, weights='distance')

knn_reg.fit(X_train_scaled, y_train)
knn_reg_Sklearn.fit(X_train_scaled, y_train)

y_pred_reg = knn_reg.predict(X_test_scaled)
y_pred_reg_Sklearn = knn_reg_Sklearn.predict(X_test_scaled)

print(f"Наша реализация R2 Score: {r2_score(y_test, y_pred_reg):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg):.3f}")
print(f"Sklearn R2 Score: {r2_score(y_test, y_pred_reg_Sklearn):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg_sklearn):.3f}")

# Линейные модели

В этом задании ты обучишь линейную регрессию, применишь регуляризацию, нормировку, проанализируешь качество модели, а также реализуешь градиентный спуск.

## Подготовка

In [ ]:
import gc

import pandas as pd
import numpy as np

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import time
from tqdm import tqdm

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler

##  Задача 1 [0,5 балла]

Объясни, почему при использовании регуляризации штрафуются только веса, но не свободный коэффициент $w_0$ (другое обозначение — $b$)?

Напомним, что предсказание линейной модели считается по формуле:

$$a(x) = \langle w, x \rangle + w_0.$$

**Напиши ответ здесь:**

-------

Для следующих заданий возьми данные из датасета [Predict Calorie Expenditure](https://www.kaggle.com/competitions/playground-series-s5e5/overview).

Цель соревнования — предсказать, сколько калорий потратит человек за тренировку по признаковому описанию как человека, так и самой тренировки.

In [ ]:
! gdown 1jb1tbxmGcgFk-jzNgu-sHXCUkgtAslRq

In [ ]:
data = pd.read_csv("/content/train.csv", index_col=0)
data.head()

В этом датасете 8 переменных:
* 1 категориальная;
* 7 вещественных.

`Calories` — целевая переменная.

Раздели датасет на признаки и целевую переменную, а также на `train` и `val`.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(data.drop('Calories', axis=1), data['Calories'], test_size=0.25, random_state=123)

## Задача 2. OneHotEncoder [0,5 балла]

В семинаре мы использовали `Target Encoding` для приведения категориальных переменных к числовому виду, так как в задаче было очень много категорий. Но для линейных моделей обычно используется `OneHotEncoding`.

Примени [OneHotEncoding из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) к категориальной переменной и получи новый датасет, который уже будет использоваться для дальнейшего обучения моделей. Но не делай этого в явном виде. Задай последовательность действий в виде [пайплайна из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). Он тебе пригодится в задачах.

Затраты по калориям не бывают отрицательными. Модель этого не знает, так как она предсказывает на всей числовой прямой. Поэтому нужно избежать отрицательных значений на этапе вызова `predict` с помощью [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).

Сделать неотрицательными предсказания калорий можно по-разному: округлить до нуля, взять минимальное по обучающей выборке.

**Задание**:
1. Преобразуй датасет к числовому виду.
2. Обучи модель линейной регрессии и сделай предсказание на валидации, чтобы проверить, что всё работает.
3. Добавь постобработку ответов.
4. Оберни все в Pipeline.

In [ ]:
# Импортируем нужные модули
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline

# Найди минимальный y
# Напиши код здесь

# Внутри Pipeline преобразуй категориальные переменные и передай результат линейной модели
pipe = # Напиши код здесь

# Обучи модель и получи предсказания для валидации, чтобы проверить работоспособность
# Напиши код здесь

y_pred_val = pipe.
y_pred_val.shape, (y_pred_val < 0).sum()

##  Задача 3. Добавь функцию оценки качества работы [1 балл]

В соревновании используется не самая стандартная метрика RMSLE (англ. Root Mean Squared Logarithmic Error).

$$RMSLE = \Big(\frac{1}{n}\sum_{i=1}^n(\log{(1+\hat y_i)} - \log{(1+y_i)})^{2}\Big)^{\frac{1}{2}}$$

Эта метрика есть в реализации [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_log_error.html), но тебе нужно написать её самостоятельно, чтобы в следующих заданиях её можно было передавать внутрь пайплайна через [make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html).

Подумай, с каким знаком должна возвращать ответ эта функция для корректной работы `make_scorer`.

**Задание:**
1. Реализуй функцию для подсчёта метрики.
2. Проверь, что она работает корректно (сравни с примером из sklearn).
3. Оцени качество предсказаний на `train` и `val`.
4. Занеси информацию о качестве в таблицу с историей.

In [ ]:
from sklearn.metrics import root_mean_squared_log_error

def scoring_foo(): # Можешь переименовать. Мы не даём корректный нейминг, потому что он подскажет знак функции
  # Напиши код здесь
  return

# Получи предсказания на train
y_pred_train = pipe. # Напиши код здесь

# Сравни ответы для валидации
print(scoring_foo(y_val, y_pred_val))
print(root_mean_squared_log_error(y_val, y_pred_val))

# Получи оценку метрики для train и val
rmsle_train = scoring_foo(y_train, y_pred_train)
rmsle_valid = scoring_foo(y_val, y_pred_val)

# Создай контейнер для хранения информации о метриках
res = {}

# Запиши метрики в контейнер и выведи таблицу
res['base'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmse_train', 'rmse_valid'])
df_res

##  Задача 4. Выбери модель и нормализацию [1,5 балла]

Используй модели:
* Ridge,
* Lasso,
* ElasticNet.

Для их качественной работы примени нормировки:
* MinMaxScaler,
* StandardScaler,
* RobustScaler.

**Задание.**
Оцени качество работы каждой пары нормировка — модель и занеси результаты в таблицу с соответствующим названием. По названию должно быть понятно, к какой комбинации относится результат.

Нормировку добавь внутрь пайплайна. За основу возьми пайплайн из задания выше.

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

# Напиши код здесь

# Выведи результат
df_res

## Задача 5. Добавь полиномы [1,5 балла]



Линейная модель может решать нелинейные задачи, если к признаковому пространству добавить их полиномы. Вряд ли сожжённые за тренировку калории линейно зависят от признаков. Поэтому добавь полиномы различных степеней к признаковому пространству, чтобы улучшить качество решения задачи.

В этом тебе поможет класс [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), он также интегрируется в пайплайн решения. Подумай, куда конкретно его вставлять в пайплайн и нужно ли добавлять полиномы OHE переменных.

Обязательно изучи документацию: там есть один аргумент, который тебе надо выключить.

Для добавления полиномов руководствуйся логикой или общими знаниями. Например, в экономических и социологических исследованиях часто добавляют к данным переменную «квадрат возраста». Её логика в следующем: человек растёт и развивается до определённого возраста, а далее угасает. Скорее всего, целевая переменная и возраст будут зависеть параболически, «квадрат возраста» позволит учесть эту параболическую зависимость.

Если не можешь придумать полиномы, то перебери переменные и значения полиномов для них. Переменных не так много, а модель учится быстро.

**Задание:**
1. Возьми лучшую по валидации комбинацию нормировка — модель.
2. Добавь к пайплайну полиномы для некоторых переменных.
3. Сохрани результат лучшего эксперимента в таблицу (у авторов задания получилось качество 0.238208).
4. Опиши, как выбирал полиномы.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Напиши код здесь

**Опиши, как ты подбирал полиномы здесь:**